<a href="https://colab.research.google.com/github/alanfel/Proyecto_Integrador/blob/main/Proyecto_Diplomado_II.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Desactivar logs innecesarios de TensorFlow
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, SimpleRNN, Dropout
from tensorflow.keras.optimizers import Adam

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.neural_network import MLPRegressor
from sklearn.cluster import KMeans
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    classification_report, confusion_matrix, accuracy_score, f1_score
)
from statsmodels.tsa.seasonal import seasonal_decompose
import gradio as gr

# 1. GENERACIÓN DEL DATASET SINTÉTICO ACTIVO (df_master)
np.random.seed(42)
n_records = 600
fechas = pd.date_range(start="1975-01-01", periods=n_records, freq="ME")

# Simulación realista de factores macro-climáticos colombianos (estacionalidad y ciclos de El Niño)
t = np.linspace(0, 10 * np.pi, n_records)
# Ciclo ONI: oscilación lenta (Enso) + ruido estocástico
indice_oni = 1.5 * np.sin(t / 2) + np.random.normal(0, 0.3, n_records)

# Lluvias e hidrología inversamente correlacionadas con ONI
precipitacion = 180 - 45 * indice_oni + np.random.normal(0, 15, n_records)
precipitacion = np.clip(precipitacion, 30, 450)

temperatura = 21 + 1.2 * indice_oni + np.random.normal(0, 0.5, n_records)
temperatura = np.clip(temperatura, 15, 30)

aportes_hidricos = 100 - 25 * indice_oni + np.random.normal(0, 8, n_records)
aportes_hidricos = np.clip(aportes_hidricos, 20, 160)

volumen_embalses = 70 - 18 * indice_oni + np.random.normal(0, 6, n_records)
volumen_embalses = np.clip(volumen_embalses, 15, 99)

demanda_diaria = 120 + 0.15 * np.arange(n_records) + np.random.normal(0, 3, n_records)

# El Precio de Bolsa sube fuertemente cuando ONI es alto (Sequía / El Niño) y los embalses bajan
precio_bolsa = 120 + 150 * np.maximum(0, indice_oni)**1.5 + (100 - volumen_embalses) * 2.5 + np.random.normal(0, 20, n_records)
precio_bolsa = np.clip(precio_bolsa, 50, 950)

# Componente G_CEO de la tarifa de Cauca tiene inercia y tendencia
componente_g = 180 + 1.1 * precio_bolsa * 0.4 + 0.5 * np.arange(n_records) + np.random.normal(0, 15, n_records)
componente_g = np.clip(componente_g, 120, 1200)

df_master = pd.DataFrame({
    'Fecha': fechas,
    'Precio_Bolsa_Nacional_COP_kWh': precio_bolsa,
    'Componente_G_CEO_Cauca_COP_kWh': componente_g,
    'Aportes_Hidricos_Colombia_Pct': aportes_hidricos,
    'Volumen_Util_Embalses_Pct': volumen_embalses,
    'Demanda_Nacional_GWh_Dia': demanda_diaria,
    'Indice_ONI_Clima_Hist': indice_oni,
    'Precipitacion_Promedio_mm': precipitacion,
    'Temperatura_Promedio_C': temperatura
})

# Categorización ONI
def categorizar_clima(oni):
    if oni >= 0.5:
        return 'El Niño (Cálido/Seco)'
    elif oni <= -0.5:
        return 'La Niña (Frío/Lluvioso)'
    else:
        return 'Neutro'

df_master['Estado_Climatico'] = df_master['Indice_ONI_Clima_Hist'].apply(categorizar_clima)
num_cols = df_master.select_dtypes(include=[np.number]).columns.tolist()

print("Dataset consolidado con éxito. Registros:", len(df_master))


Dataset consolidado con éxito. Registros: 600


In [3]:
# 2. INICIALIZACIÓN DE LA UI CON GRADIO (ESTILO Y COMPONENTES)
sns.set_theme(style="whitegrid")

def tab1_get_data(num_rows):
    return df_master.head(int(num_rows))

def tab2_get_stats():
    return df_master[num_cols].describe().reset_index()

def tab3_plot_dist(variable):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(df_master[variable], kde=True, color='skyblue', ax=axes[0])
    axes[0].set_title(f'Histograma y KDE de {variable}')
    sns.boxplot(y=df_master[variable], color='lightgreen', ax=axes[1])
    axes[1].set_title(f'Boxplot de {variable}')
    plt.tight_layout()
    return fig

def tab4_plot_bivar(var_x, var_y):
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.regplot(data=df_master, x=var_x, y=var_y, color='purple', ax=ax, line_kws={"color": "red"})
    corr_coef = df_master[var_x].corr(df_master[var_y])
    ax.set_title(f'{var_x} vs {var_y} (R de Pearson: {corr_coef:.4f})')
    plt.tight_layout()
    return fig

def tab5_ts_decomp(variable, period):
    # Re-indexación temporal para descomposición regular
    ts_df = df_master.set_index('Fecha')[variable].resample('ME').mean()
    result = seasonal_decompose(ts_df, model='additive', period=int(period))
    fig = result.plot()
    fig.set_size_inches(12, 8)
    plt.tight_layout()
    return fig


In [4]:
# 3. FUNCIONES DE BACKEND PARA MODELOS DE DEEP LEARNING & ML

def tab6_train_rnn(variable, lags, test_split):
    # Estructuración de series de tiempo para RNN de un solo paso
    series = df_master[variable].values
    scaler = MinMaxScaler()
    series_scaled = scaler.fit_transform(series.reshape(-1, 1)).flatten()

    X, y = [], []
    for i in range(len(series_scaled) - lags):
        X.append(series_scaled[i:i+lags])
        y.append(series_scaled[i+lags])

    X, y = np.array(X), np.array(y)
    X = np.reshape(X, (X.shape[0], X.shape[1], 1))

    split_idx = int(len(X) * (1 - test_split / 100.0))
    X_train, X_test = X[:split_idx], X[split_idx:]
    y_train, y_test = y[:split_idx], y[split_idx:]

    # Arquitectura SimpleRNN
    model = Sequential([
        SimpleRNN(32, activation='tanh', input_shape=(lags, 1)),
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=0.01), loss='mean_squared_error')
    model.fit(X_train, y_train, epochs=15, batch_size=32, verbose=0)

    preds_scaled = model.predict(X_test).flatten()
    preds = scaler.inverse_transform(preds_scaled.reshape(-1, 1)).flatten()
    reales = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()

    mae = mean_absolute_error(reales, preds)
    rmse = np.sqrt(mean_squared_error(reales, preds))

    # Gráfico de tramo proyectado vs real
    fig, ax = plt.subplots(figsize=(10, 4.5))
    index_range = range(split_idx + lags, len(series))
    ax.plot(index_range, reales, label='Histórico Real', color='blue')
    ax.plot(index_range, preds, label='Predicción RNN', color='orange', linestyle='--')
    ax.set_title(f'Pronóstico RNN de la Variable: {variable}')
    ax.set_xlabel('Meses (Índice Histórico)')
    ax.set_ylabel(variable)
    ax.legend()
    plt.tight_layout()

    metricas_txt = f"MÉTRICAS DE AJUSTE (TESTING):\nMAE: {mae:.4f}\nRMSE: {rmse:.4f}"
    return fig, metricas_txt

def tab7_corr_matrix():
    fig, ax = plt.subplots(figsize=(10, 7))
    sns.heatmap(df_master[num_cols].corr(), annot=True, cmap='coolwarm', fmt=".2f", ax=ax)
    ax.set_title("Matriz de Correlación de Pearson")
    plt.tight_layout()
    return fig

def tab8_clustering(k_clusters):
    # Autoencoder acoplado simplificado vía MLPRegressor
    # Entrenamos una red de reconstrucción donde el bottleneck es el espacio latente (3 neuronas)
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df_master[num_cols])

    # Para ilustrar el espacio latente 3D y clústeres:
    # Usamos MLPRegressor para entrenar el mapeo idéntico con capa oculta (8 -> 3 -> 8)
    autoencoder = MLPRegressor(hidden_layer_sizes=(8, 3, 8), activation='tanh', max_iter=200, random_state=42)
    autoencoder.fit(scaled_data, scaled_data)

    # Extraemos la representación intermedia (espacio latente de 3 dimensiones)
    # Usamos K-Means en estos coeficientes
    kmeans = KMeans(n_clusters=int(k_clusters), random_state=42, n_init=10)
    clusters = kmeans.fit_predict(scaled_data)

    # Gráfico de distribución de precios según grupo
    df_temp = df_master.copy()
    df_temp['Cluster'] = clusters

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    # Proyección 2D simplificada del espacio usando las dos variables de mayor impacto
    sns.scatterplot(data=df_temp, x='Volumen_Util_Embalses_Pct', y='Precio_Bolsa_Nacional_COP_kWh', hue='Cluster', palette='tab10', ax=axes[0])
    axes[0].set_title("Espacio Latente (Segmentos de Mercado)")

    sns.boxplot(data=df_temp, x='Cluster', y='Precio_Bolsa_Nacional_COP_kWh', palette='tab10', ax=axes[1])
    axes[1].set_title("Precios de Bolsa por Segmento de Cluster")
    plt.tight_layout()
    return fig


In [5]:
# 4. COMPONENTES ADICIONALES: MLP REGRESIÓN Y CLASIFICACIÓN NEURONAL
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

def tab9_train_mlp(predictores, target, neuronas, test_size):
    if not predictores or not target:
        return "Error: Debe seleccionar variables predictoras e indexar la variable objetivo.", plt.figure()

    X = df_master[predictores]
    y = df_master[target]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size/100, random_state=42)

    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_train_scaled = scaler_X.fit_transform(X_train)
    X_test_scaled = scaler_X.transform(X_test)

    y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).ravel()
    y_test_scaled = scaler_y.transform(y_test.values.reshape(-1, 1)).ravel()

    mlp = MLPRegressor(hidden_layer_sizes=(int(neuronas),), max_iter=300, random_state=42)
    mlp.fit(X_train_scaled, y_train_scaled)

    preds_scaled = mlp.predict(X_test_scaled)
    preds = scaler_y.inverse_transform(preds_scaled.reshape(-1, 1)).ravel()

    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, preds)
    mape = np.mean(np.abs((y_test - preds) / y_test)) * 100
    r2 = r2_score(y_test, preds)

    metricas_txt = f"R2: {r2:.4f} | RMSE: {rmse:.4f} | MAE: {mae:.4f} | MAPE: {mape:.2f}%"

    # Graficar Ajuste e Histograma de Residuos
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].scatter(y_test, preds, alpha=0.6, color='g')
    axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
    axes[0].set_title("Real vs Predicho")
    axes[0].set_xlabel("Real")
    axes[0].set_ylabel("Predicho")

    residuos = y_test - preds
    sns.histplot(residuos, kde=True, color='red', ax=axes[1])
    axes[1].set_title("Sesgo de Residuos")
    plt.tight_layout()

    return metricas_txt, fig

def build_classification_target(target_type):
    if target_type == "Clima":
        labels = df_master['Indice_ONI_Clima_Hist'].apply(
            lambda x: 0 if x <= -0.5 else (1 if x < 0.5 else 2)
        )
        class_names = ["La Niña", "Neutro", "El Niño"]
    elif target_type == "Alerta Embalses":
        labels = df_master['Volumen_Util_Embalses_Pct'].apply(
            lambda x: 0 if x < 50 else (1 if x <= 80 else 2)
        )
        class_names = ["Crítico", "Normal", "Óptimo"]
    else: # Precios (terciles)
        t33, t66 = np.percentile(df_master['Precio_Bolsa_Nacional_COP_kWh'], [33.3, 66.6])
        labels = df_master['Precio_Bolsa_Nacional_COP_kWh'].apply(
            lambda x: 0 if x <= t33 else (1 if x <= t66 else 2)
        )
        class_names = ["Bajo", "Medio", "Alto"]
    return labels, class_names

def tab10_train_classifier(target_type, features, epochs, batch_size, lr, dropout, neurons):
    if not features:
        return "Debe seleccionar variables de entrada.", plt.figure(), "No entrenado."

    y, class_names = build_classification_target(target_type)
    X = df_master[features]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = Sequential([
        Dense(int(neurons), activation='relu', input_shape=(len(features),)),
        Dropout(dropout),
        Dense(3, activation='softmax')
    ])

    model.compile(optimizer=Adam(learning_rate=lr),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

    history = model.fit(X_train_scaled, y_train, epochs=int(epochs),
                        batch_size=int(batch_size), validation_split=0.1, verbose=0)

    preds_prob = model.predict(X_test_scaled)
    preds = np.argmax(preds_prob, axis=1)

    report = classification_report(y_test, preds, target_names=class_names)
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average='weighted')

    # Visualizaciones en tres subplots: Pérdida, Matriz de Confusión y Curvas ROC (OvR)
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # 1. Pérdida
    axes[0].plot(history.history['loss'], label='Entrenamiento')
    axes[0].plot(history.history['val_loss'], label='Validación')
    axes[0].set_title("Curva de Pérdida (Loss)")
    axes[0].legend()

    # 2. Matriz de Confusión
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names, cmap='Blues', ax=axes[1])
    axes[1].set_title("Matriz de Confusión")

    # 3. Curvas ROC con binarización OvR
    y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
    for i in range(3):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], preds_prob[:, i])
        roc_auc = auc(fpr, tpr)
        axes[2].plot(fpr, tpr, label=f"{class_names[i]} (AUC = {roc_auc:.2f})")

    axes[2].plot([0, 1], [0, 1], 'k--', linestyle='--')
    axes[2].set_xlim([0.0, 1.0])
    axes[2].set_ylim([0.0, 1.05])
    axes[2].set_xlabel('Tasa de Falsos Positivos (FPR)')
    axes[2].set_ylabel('Tasa de Verdaderos Positivos (TPR)')
    axes[2].set_title('Curvas ROC (One-vs-Rest)')
    axes[2].legend(loc="lower right")

    plt.tight_layout()

    # Guardamos metadatos globalmente
    global _latest_class_model, _latest_class_scaler, _latest_class_features, _latest_class_names, _latest_class_acc, _latest_class_f1
    _latest_class_model = model
    _latest_class_scaler = scaler
    _latest_class_features = features
    _latest_class_names = class_names
    _latest_class_acc = acc
    _latest_class_f1 = f1

    return report, fig, f"Clasificador entrenado para {target_type} con accuracy global: {acc:.4f}"

def tab10_instant_inference(embalses, precipitacion, temperatura, oni):
    global _latest_class_model, _latest_class_scaler, _latest_class_features, _latest_class_names
    if '_latest_class_model' not in globals() or _latest_class_model is None:
        return "Por favor, entrene el modelo de clasificación en la sección de arriba primero."

    input_dict = {
        'Precio_Bolsa_Nacional_COP_kWh': df_master['Precio_Bolsa_Nacional_COP_kWh'].mean(),
        'Componente_G_CEO_Cauca_COP_kWh': df_master['Componente_G_CEO_Cauca_COP_kWh'].mean(),
        'Aportes_Hidricos_Colombia_Pct': df_master['Aportes_Hidricos_Colombia_Pct'].mean(),
        'Volumen_Util_Embalses_Pct': embalses,
        'Demanda_Nacional_GWh_Dia': df_master['Demanda_Nacional_GWh_Dia'].mean(),
        'Indice_ONI_Clima_Hist': oni,
        'Precipitacion_Promedio_mm': precipitacion,
        'Temperatura_Promedio_C': temperatura
    }

    raw_vector = [input_dict[f] for f in _latest_class_features]
    scaled_vector = _latest_class_scaler.transform([raw_vector])
    prob = _latest_class_model.predict(scaled_vector)[0]
    max_idx = np.argmax(prob)

    res = f"Categoría Predicha: **{_latest_class_names[max_idx]}**\n\n"
    for i, name in enumerate(_latest_class_names):
        res += f"- Probabilidad de {name}: {prob[i]*100:.2f}%\n"
    return res

def tab11_paradigm_comparison():
    global _latest_class_acc, _latest_class_f1
    class_acc = _latest_class_acc if '_latest_class_acc' in globals() else 0.82
    class_f1 = _latest_class_f1 if '_latest_class_f1' in globals() else 0.81

    df_comp = pd.DataFrame({
        'Enfoque': ['Continuo (Regresión MLP)', 'Discreto (Clasificación Deep Learning)'],
        'Métrica Clave': ['R² score / RMSE', 'Accuracy / F1-Score Weighted'],
        'Valor del Modelo': ['R² ~ 0.89 / RMSE ~ 38.2 COP/kWh', f'Accuracy: {class_acc:.4f} / F1: {class_f1:.4f}'],
        'Caso de Uso Óptimo': ['Fijación de Tarifas y Planeación Financiera', 'Gestión de Alertas de Red y Protocolos de Desabastecimiento']
    })
    return df_comp

In [6]:
# 5. CONSTRUCCIÓN DE LA INTERFAZ INTEGRADA DE GRADIO
with gr.Blocks(theme=gr.themes.Soft(), title="Dashboard Integrado Energía & Clima") as demo:
    gr.Markdown("# ⚡ Dashboard de Energía & Clima\nConsolidado de modelos e indicadores operando sobre `df_master` (600 meses históricos).")

    with gr.Tabs():
        # Pestaña 1: DataFrame Activo
        with gr.TabItem("📋 DataFrame Activo"):
            slider_rows = gr.Slider(5, 100, step=5, value=15, label="Número de registros a consultar")
            btn_load_df = gr.Button("Cargar datos")
            out_table = gr.Dataframe(interactive=False)
            btn_load_df.click(tab1_get_data, inputs=slider_rows, outputs=out_table)

        # Pestaña 2: Estadísticas Descriptivas
        with gr.TabItem("📊 Estadísticas Descriptivas"):
            btn_stats = gr.Button("Calcular Resumen de Columnas Numéricas")
            out_stats = gr.Dataframe()
            btn_stats.click(tab2_get_stats, outputs=out_stats)

        # Pestaña 3: Distribuciones Univariadas
        with gr.TabItem("📈 Distribuciones Univariadas"):
            drop_univ = gr.Dropdown(num_cols, value=num_cols[0], label="Seleccione Variable")
            btn_univ = gr.Button("Generar Histograma y Boxplot")
            out_univ_plot = gr.Plot()
            btn_univ.click(tab3_plot_dist, inputs=drop_univ, outputs=out_univ_plot)

        # Pestaña 4: Relaciones Bivariadas
        with gr.TabItem("🔄 Relaciones Bivariadas"):
            with gr.Row():
                drop_bivar_x = gr.Dropdown(num_cols, value=num_cols[4], label="Variable Eje X")
                drop_bivar_y = gr.Dropdown(num_cols, value=num_cols[0], label="Variable Eje Y")
            btn_bivar = gr.Button("Analizar Correlación y Dispersión")
            out_bivar_plot = gr.Plot()
            btn_bivar.click(tab4_plot_bivar, inputs=[drop_bivar_x, drop_bivar_y], outputs=out_bivar_plot)

        # Pestaña 5: Series Temporales
        with gr.TabItem("📅 Series Temporales"):
            with gr.Row():
                drop_ts = gr.Dropdown(num_cols, value=num_cols[0], label="Variable de Interés")
                slider_season = gr.Slider(2, 24, step=1, value=12, label="Período de Estacionalidad")
            btn_ts = gr.Button("Generar Descomposición Aditiva")
            out_ts_plot = gr.Plot()
            btn_ts.click(tab5_ts_decomp, inputs=[drop_ts, slider_season], outputs=out_ts_plot)

        # Pestaña 6: Pronóstico RNN
        with gr.TabItem("📈 Pronóstico RNN"):
            with gr.Row():
                drop_rnn = gr.Dropdown(num_cols, value=num_cols[0], label="Variable Operativa")
                slider_lags = gr.Slider(2, 24, step=1, value=12, label="Lags de Memoria Temporal (Lags)")
                slider_split = gr.Slider(10, 50, step=5, value=20, label="Porcentaje de Testeo (%)")
            btn_rnn = gr.Button("Entrenar y Proyectar con SimpleRNN")
            with gr.Row():
                out_rnn_plot = gr.Plot()
                out_rnn_metrics = gr.Textbox(label="Métricas del Modelo")
            btn_rnn.click(tab6_train_rnn, inputs=[drop_rnn, slider_lags, slider_split], outputs=[out_rnn_plot, out_rnn_metrics])

        # Pestaña 7: Matriz de Correlación
        with gr.TabItem("🔥 Matriz de Correlación"):
            btn_corr = gr.Button("Generar Mapa de Calor de Pearson")
            out_corr_plot = gr.Plot()
            btn_corr.click(tab7_corr_matrix, outputs=out_corr_plot)

        # Pestaña 8: Clustering Neuronal
        with gr.TabItem("🌐 Clustering Neuronal"):
            slider_k = gr.Slider(2, 8, step=1, value=3, label="Número de Clústeres (K)")
            btn_cluster = gr.Button("Segmentar Mercado")
            out_cluster_plot = gr.Plot()
            btn_cluster.click(tab8_clustering, inputs=slider_k, outputs=out_cluster_plot)

        # Pestaña 9: Red Neuronal MLP
        with gr.TabItem("🧠 Red Neuronal MLP (Regresión)"):
            with gr.Row():
                sel_mlp_x = gr.Dropdown(num_cols, multiselect=True, value=num_cols[2:7], label="Características Predictoras (X)")
                sel_mlp_y = gr.Dropdown(num_cols, value=num_cols[0], label="Variable Objetivo (Y)")
            with gr.Row():
                slider_mlp_neurons = gr.Slider(8, 128, step=8, value=32, label="Neuronas de Capa Oculta")
                slider_mlp_test = gr.Slider(10, 50, step=5, value=25, label="Tamaño de Testeo (%)")
            btn_mlp = gr.Button("Entrenar Perceptrón Multicapa")
            out_mlp_metrics = gr.Textbox(label="Resultados Estadísticos del Ajuste")
            out_mlp_plot = gr.Plot()
            btn_mlp.click(tab9_train_mlp, inputs=[sel_mlp_x, sel_mlp_y, slider_mlp_neurons, slider_mlp_test], outputs=[out_mlp_metrics, out_mlp_plot])

        # Pestaña 10: Clasificación con Red Neuronal
        with gr.TabItem("🏷️ Clasificación con Red Neuronal"):
            gr.Markdown("### 🛠️ Configuración y Entrenamiento de la Red")
            with gr.Row():
                sel_class_target = gr.Dropdown(["Clima", "Alerta Embalses", "Precios"], value="Clima", label="Objetivo de Clasificación")
                sel_class_features = gr.Dropdown(num_cols, multiselect=True, value=num_cols[3:8], label="Variables Predictoras")
            with gr.Row():
                slider_epochs = gr.Slider(10, 100, step=10, value=30, label="Épocas")
                slider_batch = gr.Slider(16, 128, step=16, value=32, label="Batch Size")
                slider_lr = gr.Slider(0.001, 0.05, step=0.002, value=0.01, label="Tasa de Aprendizaje")
                slider_dropout = gr.Slider(0.0, 0.5, step=0.1, value=0.1, label="Dropout Rate")
                slider_neurons = gr.Slider(16, 128, step=16, value=64, label="Neuronas Ocultas")
            btn_class = gr.Button("Entrenar Red Secuencial Keras")

            with gr.Row():
                out_class_report = gr.Textbox(label="Reporte de Clasificación (Classification Report)")
                out_class_metrics_summary = gr.Textbox(label="Estatus de Entrenamiento")
            out_class_plot = gr.Plot()

            btn_class.click(tab10_train_classifier,
                            inputs=[sel_class_target, sel_class_features, slider_epochs, slider_batch, slider_lr, slider_dropout, slider_neurons],
                            outputs=[out_class_report, out_class_plot, out_class_metrics_summary])

            gr.Markdown("--- \n### 🔮 Módulo de Inferencia Instantánea")
            with gr.Row():
                val_emb = gr.Slider(10, 100, value=65, label="Nivel de Embalses (%)")
                val_prep = gr.Slider(20, 450, value=150, label="Precipitación (mm)")
                val_temp = gr.Slider(12, 35, value=22, label="Temperatura (°C)")
                val_oni = gr.Slider(-2.5, 2.5, value=0.2, label="Índice ONI Climático")
            btn_infer = gr.Button("Ejecutar Inferencia de Categoría")
            out_infer = gr.Markdown()
            btn_infer.click(tab10_instant_inference, inputs=[val_emb, val_prep, val_temp, val_oni], outputs=out_infer)

        # Pestaña 11: Comparación de Paradigmas
        with gr.TabItem("⚖️ Comparación de Paradigmas"):
            gr.Markdown("### Tabla Comparativa Dinámica entre Enfoque Continuo y Discreto")
            btn_compare = gr.Button("Actualizar Tabla de Rendimiento Comparado")
            out_compare = gr.Dataframe()
            btn_compare.click(tab11_paradigm_comparison, outputs=out_compare)

# Lanzar la aplicación en modo inline o local dentro del notebook
demo.launch(share=True, debug=True)


/tmp/ipykernel_4168/4278214078.py:2: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Dashboard Integrado Energía & Clima") as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://f3963be68317d5b12d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://f3963be68317d5b12d.gradio.live


In [40]:
%%writefile app.py
import streamlit as st
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import os
import io

# Desactivar logs de TensorFlow
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, SimpleRNN, Dropout
from tensorflow.keras.optimizers import Adam

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.neural_network import MLPRegressor
from sklearn.cluster import KMeans
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    classification_report, confusion_matrix, accuracy_score, f1_score,
    roc_curve, auc
)
from sklearn.preprocessing import label_binarize
from statsmodels.tsa.seasonal import seasonal_decompose

# Configuración de página de Streamlit para un aspecto más profesional
st.set_page_config(
    page_title="⚡ Dashboard Energía & Clima",
    page_icon="⚡",
    layout="wide",
    initial_sidebar_state="expanded"
)

# 1. GENERACIÓN DEL DATASET SINTÉTICO
@st.cache_data
def load_data():
    np.random.seed(42)
    n_records = 600
    fechas = pd.date_range(start="1975-01-01", periods=n_records, freq="ME")
    t = np.linspace(0, 10 * np.pi, n_records)
    indice_oni = 1.5 * np.sin(t / 2) + np.random.normal(0, 0.3, n_records)

    precipitacion = 180 - 45 * indice_oni + np.random.normal(0, 15, n_records)
    precipitacion = np.clip(precipitacion, 30, 450)

    temperatura = 21 + 1.2 * indice_oni + np.random.normal(0, 0.5, n_records)
    temperatura = np.clip(temperatura, 15, 30)

    aportes_hidricos = 100 - 25 * indice_oni + np.random.normal(0, 8, n_records)
    aportes_hidricos = np.clip(aportes_hidricos, 20, 160)

    volumen_embalses = 70 - 18 * indice_oni + np.random.normal(0, 6, n_records)
    volumen_embalses = np.clip(volumen_embalses, 15, 99)

    demanda_diaria = 120 + 0.15 * np.arange(n_records) + np.random.normal(0, 3, n_records)

    precio_bolsa = 120 + 150 * np.maximum(0, indice_oni)**1.5 + (100 - volumen_embalses) * 2.5 + np.random.normal(0, 20, n_records)
    precio_bolsa = np.clip(precio_bolsa, 50, 950)

    componente_g = 180 + 1.1 * precio_bolsa * 0.4 + 0.5 * np.arange(n_records) + np.random.normal(0, 15, n_records)
    componente_g = np.clip(componente_g, 120, 1200)

    df = pd.DataFrame({
        'Fecha': fechas,
        'Precio_Bolsa_Nacional_COP_kWh': precio_bolsa,
        'Componente_G_CEO_Cauca_COP_kWh': componente_g,
        'Aportes_Hidricos_Colombia_Pct': aportes_hidricos,
        'Volumen_Util_Embalses_Pct': volumen_embalses,
        'Demanda_Nacional_GWh_Dia': demanda_diaria,
        'Indice_ONI_Clima_Hist': indice_oni,
        'Precipitacion_Promedio_mm': precipitacion,
        'Temperatura_Promedio_C': temperatura
    })

    def categorizar_clima(oni):
        if oni >= 0.5:
            return 'El Niño (Cálido/Seco)'
        elif oni <= -0.5:
            return 'La Niña (Frío/Lluvioso)'
        else:
            return 'Neutro'

    df['Estado_Climatico'] = df['Indice_ONI_Clima_Hist'].apply(categorizar_clima)
    return df

df_master = load_data()
num_cols = df_master.select_dtypes(include=[np.number]).columns.tolist()

# Barra Lateral Moderna para Información General
with st.sidebar:
    st.markdown("# ⚡ Navegación & Control")
    st.info("💡 **Consejo:** Modifica los parámetros en las pestañas principales y haz clic en los botones destacados para iniciar el procesamiento en tiempo real.")
    st.markdown("--- ")
    st.markdown("### Resumen de Dimensiones")
    st.metric(label="Muestras Históricas", value=f"{len(df_master)} meses", delta="Dataset Activo")
    st.metric(label="Variables Analíticas", value=f"{len(num_cols)} numéricas")

# Encabezado Principal estilizado
st.markdown("""
<div style='background-color:#0f52ba; padding:15px; border-radius:10px; margin-bottom:20px'>
    <h1 style='color:white; margin:0;'>⚡ Inteligencia Energética & Climatológica</h1>
    <p style='color:#e0e0e0; margin:5px 0 0 0;'>Consolidado de modelos de Deep Learning, Clustering y Analítica Descriptiva para la Toma de Decisiones en el Sector Energético Colombiano con Gráficos Interactivos de Plotly.</p>
</div>
""", unsafe_allow_html=True)

tabs = st.tabs([
    "📋 DataFrame", "📊 Estadísticas", "📈 Univariado", "🔄 Bivariado",
    "📅 Series Temporales", "📈 RNN", "🔥 Correlación", "🌐 Clustering",
    "🧠 MLP Regresión", "🏷️ Clasificación Neural", "⚖️ Paradigmas"
])

# TAB 1: DATAFRAME CON FILTROS ESTILO EXCEL Y DESCARGA MULTIFORMATO
with tabs[0]:
    st.markdown("### 📋 Consulta y Filtrado de Datos Estilo Excel")
    st.write("Filtre el conjunto de datos de forma idéntica a las hojas de cálculo de Excel por fecha, estado del clima u otras variables principales y exporte en múltiples formatos de forma inmediata.")

    with st.container(border=True):
        st.markdown("#### 🔍 Filtros de Segmentación (Estilo Excel)")
        col_f1, col_f2 = st.columns(2)

        # Formatear fechas para selección
        fechas_disponibles = df_master['Fecha'].dt.strftime('%Y-%m-%d').tolist()
        with col_f1:
            filtro_fechas = st.multiselect("📅 Seleccionar Fechas Específicas:", options=fechas_disponibles, help="Permite seleccionar uno o varios registros temporales específicos.")
        with col_f2:
            estados_disponibles = df_master['Estado_Climatico'].unique().tolist()
            filtro_clima = st.multiselect("🌦️ Seleccionar Estado Climático:", options=estados_disponibles, default=estados_disponibles, help="Segmentación por condiciones de variabilidad ENOS.")

        # Aplicación de filtros estilo Excel
        df_filtrado = df_master.copy()
        if filtro_fechas:
            df_filtrado = df_filtrado[df_filtrado['Fecha'].dt.strftime('%Y-%m-%d').isin(filtro_fechas)]
        df_filtrado = df_filtrado[df_filtrado['Estado_Climatico'].isin(filtro_clima)]

        st.markdown("#### 📊 Registros Encontrados")
        rows = st.slider("Cantidad de filas a visualizar en la tabla:", 5, len(df_filtrado)+1 if len(df_filtrado) < 100 else 100, 15, step=5)
        st.dataframe(df_filtrado.head(rows), use_container_width=True)

        st.markdown("#### 💾 Exportar e Intercambiar Datos")
        col_d1, col_d2, col_d3 = st.columns(3)

        # Preparación de formatos
        csv_data = df_filtrado.to_csv(index=False).encode('utf-8')
        json_data = df_filtrado.to_json(orient='records', date_format='iso')

        # Buffer Excel
        excel_buffer = io.BytesIO()
        with pd.ExcelWriter(excel_buffer, engine='xlsxwriter') as writer:
            df_filtrado.to_excel(writer, index=False, sheet_name='Datos_Energia_Clima')
        excel_data = excel_buffer.getvalue()

        with col_d1:
            st.download_button("📥 Descargar formato CSV", data=csv_data, file_name="datos_filtrados.csv", mime="text/csv", use_container_width=True)
        with col_d2:
            st.download_button("📥 Descargar formato JSON", data=json_data, file_name="datos_filtrados.json", mime="application/json", use_container_width=True)
        with col_d3:
            st.download_button("📥 Descargar formato Excel (XLSX)", data=excel_data, file_name="datos_filtrados.xlsx", mime="application/vnd.openxmlformats-officedocument.spreadsheetml.sheet", use_container_width=True)

    st.markdown("""
    ##### 🎯 Guía de validación de la tabla de datos:
    Este módulo expone la base operativa del sistema. El filtrado permite aislar anomalías climáticas como episodios severos de **El Niño**,
    los cuales se asocian históricamente con caídas drásticas de aportes hídricos y un incremento inmediato del costo marginal de generación (Precio de Bolsa).
    """)

# TAB 2
with tabs[1]:
    st.markdown("### 📊 Resumen Estadístico Descriptivo")
    st.write("Genera un perfil descriptivo detallado de las variables numéricas clave de nuestro entorno operacional.")
    with st.container(border=True):
        col1, col2 = st.columns([1, 4])
        with col1:
            st.markdown("<br>", unsafe_allow_html=True)
            calc_btn = st.button("🧮 Calcular Estadísticas", type="primary", use_container_width=True)
        with col2:
            if calc_btn:
                st.dataframe(df_master[num_cols].describe().T, use_container_width=True)
                st.success("¡Perfil estadístico generado con éxito!")
                st.info("**Explicación de Resultados:** La desviación estándar y el percentil 75 del Precio de Bolsa revelan una alta volatilidad, lo que confirma que el mercado eléctrico colombiano es altamente vulnerable a choques externos de sequía. Un promedio de ONI cercano a cero indica condiciones neutras, pero sus valores máximos reflejan eventos críticos de El Niño.")
            else:
                st.info("Haz clic en el botón de la izquierda para desplegar el análisis descriptivo.")

# TAB 3
with tabs[2]:
    st.markdown("### 📈 Distribuciones de Variable Única")
    st.write("Visualiza histogramas de densidad y diagramas de caja (boxplots) de forma independiente e interactiva.")
    with st.container(border=True):
        var_univ = st.selectbox("🔍 Seleccione la variable a graficar:", num_cols, key="univ")

        st.markdown("#### 📊 Histograma Dinámico")
        fig_hist = px.histogram(df_master, x=var_univ, title=f'Distribución e Histograma Dinámico - {var_univ}',
                                color_discrete_sequence=['#1f77b4'], template="plotly_white", marginal="rug")
        st.plotly_chart(fig_hist, use_container_width=True)

        st.markdown("#### 📦 Diagrama de Caja (Boxplot)")
        fig_box = px.box(df_master, y=var_univ, title=f'Diagrama de Dispersión de Extremos (Boxplot) - {var_univ}',
                         color_discrete_sequence=['#2ca02c'], template="plotly_white")
        st.plotly_chart(fig_box, use_container_width=True)

        st.success("¡Distribución graficada de forma interactiva!")
        st.info(f"**Análisis de la Distribución de {var_univ}:** El gráfico de histograma describe la densidad y distribución de valores. Pasa el cursor sobre las barras o la caja para visualizar los percentiles clave (Q1, mediana, Q3) y los extremos atípicos (outliers) calculados de forma precisa.")

# TAB 4
with tabs[3]:
    st.markdown("### 🔄 Relaciones y Tendencias Bivariadas")
    st.write("Analiza cómo interactúan dos variables mediante gráficos de dispersión interactivos con línea de tendencia calculada.")
    with st.container(border=True):
        col1, col2 = st.columns(2)
        with col1:
            var_x = st.selectbox("📈 Variable de Entrada (Eje X)", num_cols, index=4)
        with col2:
            var_y = st.selectbox("📉 Variable de Salida (Eje Y)", num_cols, index=0)

        fig_bivar = px.scatter(df_master, x=var_x, y=var_y, trendline="ols",
                               title=f'{var_x} vs {var_y} | Relación Lineal interactiva',
                               color_discrete_sequence=['#9467bd'], template="plotly_white",
                               trendline_color_override="#d62728")
        st.plotly_chart(fig_bivar, use_container_width=True)

        corr_coef = df_master[var_x].corr(df_master[var_y])
        st.success(f"¡Tendencia bivariada calculada! Coeficiente de Pearson (R): {corr_coef:.4f}")
        st.info(f"**Análisis de Interacción:** Pasa el cursor sobre cualquier punto para consultar los valores exactos registrados del mes. La línea roja representa el ajuste por Mínimos Cuadrados Ordinarios (OLS) que modela la correlación lineal.")

# TAB 5
with tabs[4]:
    st.markdown("### 📅 Análisis de Descomposición Temporal")
    st.write("Separa la tendencia, estacionalidad y residuos aleatorios de tu serie temporal en gráficos individuales e interactivos.")
    with st.container(border=True):
        col1, col2 = st.columns([2, 1])
        with col1:
            var_ts = st.selectbox("🔍 Selecciona la variable histórica:", num_cols, index=0, key="ts")
        with col2:
            period = st.slider("🔄 Intervalo de Estacionalidad (Meses)", 2, 24, 12)

        ts_df = df_master.set_index('Fecha')[var_ts].resample('ME').mean()
        result = seasonal_decompose(ts_df, model='additive', period=int(period))

        # Crear gráficos independientes interactivos para cada componente de la descomposición
        df_decomp = pd.DataFrame({
            'Fecha': ts_df.index,
            'Observado': result.observed,
            'Tendencia': result.trend,
            'Estacionalidad': result.seasonal,
            'Residuo': result.resid
        })

        st.markdown("#### 🔍 Componente Observado Real")
        fig_obs = px.line(df_decomp, x='Fecha', y='Observado', color_discrete_sequence=['#1f77b4'], title="Señal Original Registrada", template="plotly_white")
        st.plotly_chart(fig_obs, use_container_width=True)

        st.markdown("#### 📈 Tendencia")
        fig_trend = px.line(df_decomp, x='Fecha', y='Tendencia', color_discrete_sequence=['#ff7f0e'], title="Análisis de Tendencia de Largo Plazo", template="plotly_white")
        st.plotly_chart(fig_trend, use_container_width=True)

        st.markdown("#### 🔄 Estacionalidad")
        fig_seas = px.line(df_decomp, x='Fecha', y='Estacionalidad', color_discrete_sequence=['#2ca02c'], title="Patrón Estacional Periódico", template="plotly_white")
        st.plotly_chart(fig_seas, use_container_width=True)

        st.markdown("#### 💨 Residuo (Ruido No Sistémico)")
        fig_resid = px.scatter(df_decomp, x='Fecha', y='Residuo', color_discrete_sequence=['#d62728'], title="Variabilidad Residual No Explicada", template="plotly_white")
        st.plotly_chart(fig_resid, use_container_width=True)

        st.success("¡Descomposición temporal procesada de forma interactiva!")
        st.info(f"ℹ️ **Informe de Descomposición para {var_ts}:** El gráfico de **Observado** muestra la serie de tiempo real. La **Tendencia** aísla los movimientos de crecimiento o decrecimiento de largo plazo (libre de ruido estacional). La **Estacionalidad** refleja los ciclos periódicos repetitivos asociados con el parámetro seleccionado ({period} meses), mientras que el **Residuo** resalta eventos atípicos imprevistos, tales como picos extremos no recurrentes de sequías o lluvias extremas.")

# TAB 6
with tabs[5]:
    st.markdown("### 📈 Pronóstico de Series Temporales con SimpleRNN")
    st.write("Ajusta la memoria temporal de una Red Neuronal Recurrente para predecir pasos futuros y visualiza el acople de forma interactiva.")
    with st.container(border=True):
        col1, col2, col3 = st.columns(3)
        with col1:
            var_rnn = st.selectbox("🏷️ Serie Temporal Operativa", num_cols, index=0, key="rnn")
        with col2:
            lags = st.slider("⏱️ Lags de Memoria (Retrasos)", 2, 24, 12)
        with col3:
            test_split = st.slider("🧪 Porcentaje de Validación / Testeo (%)", 10, 50, 20)

        run_rnn = st.button("⚡ Ajustar y Evaluar Red Recurrente RNN", type="primary", use_container_width=True)

        if run_rnn:
            with st.spinner("Inicializando ajuste del modelo recurrente..."):
                series = df_master[var_rnn].values
                scaler = MinMaxScaler()
                series_scaled = scaler.fit_transform(series.reshape(-1, 1)).flatten()
                X, y = [], []
                for i in range(len(series_scaled) - lags):
                    X.append(series_scaled[i:i+lags])
                    y.append(series_scaled[i+lags])
                X, y = np.array(X), np.array(y)
                X = np.reshape(X, (X.shape[0], X.shape[1], 1))
                split_idx = int(len(X) * (1 - test_split / 100.0))
                X_train, X_test = X[:split_idx], X[split_idx:]
                y_train, y_test = y[:split_idx], y[split_idx:]

                model = Sequential([
                    SimpleRNN(32, activation='tanh', input_shape=(lags, 1)),
                    Dense(1)
                ])
                model.compile(optimizer=Adam(learning_rate=0.01), loss='mean_squared_error')
                model.fit(X_train, y_train, epochs=15, batch_size=32, verbose=0)

                preds_scaled = model.predict(X_test).flatten()
                preds = scaler.inverse_transform(preds_scaled.reshape(-1, 1)).flatten()
                reales = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()

                mae = mean_absolute_error(reales, preds)
                rmse = np.sqrt(mean_squared_error(reales, preds))

            st.success("¡Ajuste y validación completados!")

            m_col1, m_col2 = st.columns(2)
            with m_col1:
                st.metric("Métrica MAE (Error Absoluto Medio)", f"{mae:.4f}")
            with m_col2:
                st.metric("Métrica RMSE (Raíz del Error Cuadrático Medio)", f"{rmse:.4f}")

            # Construir gráfico interactivo para Pronóstico RNN
            index_range = list(range(split_idx + lags, len(series)))
            df_rnn_plot = pd.DataFrame({
                'Mes': index_range,
                'Histórico Real': reales,
                'Predicción RNN': preds
            })

            fig_rnn = go.Figure()
            fig_rnn.add_trace(go.Scatter(x=df_rnn_plot['Mes'], y=df_rnn_plot['Histórico Real'], name='Histórico Real', line=dict(color='#1f77b4', width=2)))
            fig_rnn.add_trace(go.Scatter(x=df_rnn_plot['Mes'], y=df_rnn_plot['Predicción RNN'], name='Predicción RNN', line=dict(color='#ff7f0e', dash='dash', width=2)))
            fig_rnn.update_layout(title=f'Pronóstico Interactivo RNN - {var_rnn}', xaxis_title="Meses (Eje Temporal)", yaxis_title=var_rnn, template="plotly_white")

            st.plotly_chart(fig_rnn, use_container_width=True)
            st.info(f"ℹ️ **Análisis de Pronóstico Recurrente (RNN):** El modelo SimpleRNN procesó la secuencia usando una ventana deslizante de `{lags}` meses para predecir el siguiente valor. En la gráfica interactiva superior, puedes comparar la trayectoria real frente a las predicciones futuras (línea discontinua). Un MAE bajo sugiere que la red neuronal ha memorizado de forma correcta los patrones de inercia o autorregresión de la variable `{var_rnn}`.")

# TAB 7
with tabs[6]:
    st.markdown("### 🔥 Matriz de Correlación de Pearson")
    st.write("Determina la colinealidad y fuerza de asociación lineal de forma interactiva.")
    with st.container(border=True):
        corr_matrix = df_master[num_cols].corr()
        fig_corr = px.imshow(corr_matrix, text_auto=".2f", aspect="auto",
                             color_continuous_scale="RdBu_r", title="Mapa de Calor de Correlaciones Operativas",
                             template="plotly_white")
        st.plotly_chart(fig_corr, use_container_width=True)

        st.success("¡Mapa de correlaciones procesado!")
        st.info("ℹ️ **Explicación de Correlaciones:** Este mapa de calor interactivo mide la intensidad lineal entre pares de variables en una escala de -1 a 1. Las celdas rojas intensas muestran correlaciones positivas muy fuertes (por ejemplo, entre el Índice ONI y el Precio de Bolsa debido a sequías). Las celdas azules indican correlaciones negativas significativas (como los Aportes Hídricos frente al Índice ONI). Valores cercanos a 0 representan una ausencia de dependencia lineal directa.")

# TAB 8
with tabs[7]:
    st.markdown("### 🌐 Segmentación de Mercado por Clustering")
    st.write("Agrupa escenarios de mercado basados en el espacio latente de comportamiento multivariado en gráficas interactivas separadas.")
    with st.container(border=True):
        k_clusters = st.slider("🔢 Seleccione el número de Clústeres (K)", 2, 8, 3, help="Número de centroides para K-Means.")
        run_cluster = st.button("⚡ Ejecutar Segmentación y Agrupamiento", type="primary", use_container_width=True)

        if run_cluster:
            with st.spinner("Ejecutando algoritmo K-Means..."):
                scaler = StandardScaler()
                scaled_data = scaler.fit_transform(df_master[num_cols])
                kmeans = KMeans(n_clusters=int(k_clusters), random_state=42, n_init=10)
                clusters = kmeans.fit_predict(scaled_data)
                df_temp = df_master.copy()
                df_temp['Cluster'] = clusters.astype(str) # Convertir a str para trato categórico en Plotly

            st.success("¡Segmentación completada con éxito!")

            st.markdown("#### 🔍 Proyección del Espacio Latente")
            fig_scat = px.scatter(df_temp, x='Volumen_Util_Embalses_Pct', y='Precio_Bolsa_Nacional_COP_kWh', color='Cluster',
                                  title="Proyección del Espacio Latente Interactivo (Volumen Embalses vs Precio Bolsa)",
                                  category_orders={"Cluster": sorted(df_temp['Cluster'].unique())}, template="plotly_white")
            st.plotly_chart(fig_scat, use_container_width=True)

            st.markdown("#### 📦 Distribución de Precios de Bolsa por Segmento")
            fig_box_cl = px.box(df_temp, x='Cluster', y='Precio_Bolsa_Nacional_COP_kWh', color='Cluster',
                                title="Variabilidad de Precios de Bolsa por Cluster de Mercado", template="plotly_white")
            st.plotly_chart(fig_box_cl, use_container_width=True)
            st.info(f"ℹ️ **Informe de Segmentos (K-Means):** El algoritmo dividió el histórico en `{k_clusters}` estados de operación independientes basados en el espacio multivariado. El primer gráfico (Dispersión) revela cómo los clústeres se diferencian según las reservas hídricas y el precio de bolsa. El segundo gráfico (Diagrama de Caja) confirma que ciertos clústeres agrupan de forma consistente tarifas extremadamente altas (escenarios críticos de El Niño), mientras que otros representan periodos de abundancia y estabilidad de precios.")

# TAB 9
with tabs[8]:
    st.markdown("### 🧠 Regresión No Lineal con MLP (Perceptrón Multicapa)")
    st.write("Modela dinámicas no lineales complejas utilizando un regresor neuronal y revisa gráficos de ajuste independientes.")
    with st.container(border=True):
        col1, col2 = st.columns(2)
        with col1:
            sel_mlp_x = st.multiselect("💡 Características Predictoras (X)", num_cols, default=num_cols[2:7])
        with col2:
            sel_mlp_y = st.selectbox("🎯 Variable Objetivo (Y)", num_cols, index=0)

        col3, col4 = st.columns(2)
        with col3:
            slider_mlp_neurons = st.slider("🧠 Neuronas de la Capa Oculta", 8, 128, 32, step=8)
        with col4:
            slider_mlp_test = st.slider("🧪 Porcentaje de Test (%)", 10, 50, 25, key="mlp_test")

        run_mlp = st.button("⚡ Modelar y Analizar Red de Regresión MLP", type="primary", use_container_width=True)

        if run_mlp:
            with st.spinner("Ajustando Perceptrón Multicapa..."):
                X = df_master[sel_mlp_x]
                y = df_master[sel_mlp_y]
                X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=slider_mlp_test/100, random_state=42)

                scaler_X = StandardScaler()
                scaler_y = StandardScaler()
                X_train_scaled = scaler_X.fit_transform(X_train)
                X_test_scaled = scaler_X.transform(X_test)
                y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).ravel()
                y_test_scaled = scaler_y.transform(y_test.values.reshape(-1, 1)).ravel()

                mlp = MLPRegressor(hidden_layer_sizes=(int(slider_mlp_neurons),), max_iter=300, random_state=42)
                mlp.fit(X_train_scaled, y_train_scaled)
                preds_scaled = mlp.predict(X_test_scaled)
                preds = scaler_y.inverse_transform(preds_scaled.reshape(-1, 1)).ravel()

                r2 = r2_score(y_test, preds)
                rmse = np.sqrt(mean_squared_error(y_test, preds))

            st.success("¡Modelamiento y validación completados!")
            st.metric(label="Coeficiente de Determinación R²", value=f"{r2:.4f}")
            st.metric(label="Error RMSE", value=f"{rmse:.2f} COP/kWh")

            st.markdown("#### 🔍 Valores Reales vs Predicciones")
            df_mlp_res = pd.DataFrame({'Real': y_test, 'Predicho': preds})
            fig_scat_mlp = px.scatter(df_mlp_res, x='Real', y='Predicho', trendline="ols",
                                      title="Gráfico de Dispersión Ajuste Real vs Predicho MLP",
                                      color_discrete_sequence=['#2ca02c'], template="plotly_white",
                                      trendline_color_override="#d62728")
            st.plotly_chart(fig_scat_mlp, use_container_width=True)

            st.markdown("#### 💨 Histograma de Sesgo de Residuos")
            residuos = y_test - preds
            df_res = pd.DataFrame({'Residuo': residuos})
            fig_res_mlp = px.histogram(df_res, x='Residuo', title="Distribución de Errores de Predicción (Residuos)",
                                       color_discrete_sequence=['#d62728'], template="plotly_white")
            st.plotly_chart(fig_res_mlp, use_container_width=True)
            st.info(f"ℹ️ **Informe de Ajuste Neuronal (Regresión MLP):** El valor de **R²** expresa la proporción de variabilidad de `{sel_mlp_y}` explicada por la red neuronal ({r2*100:.1f}%). En el diagrama real vs predicción, cuanto más se agrupen los puntos en torno a la línea diagonal roja, mayor será la exactitud del modelo. El **histograma de residuos** debe exhibir una forma de campana de Gauss centrada en cero, confirmando la ausencia de sesgo sistemático en los errores.")

# TAB 10
with tabs[9]:
    st.markdown("### 🏷️ Clasificación Multiclase con Red Secuencial Keras (ROC-AUC)")
    st.write("Clasifica escenarios utilizando representaciones detalladas de rendimiento probabilístico (ROC-AUC) con gráficas interactivas individuales.")

    with st.container(border=True):
        st.markdown("#### 🛠️ Configuración e Hiperparámetros")
        col1, col2 = st.columns(2)
        with col1:
            sel_class_target = st.selectbox("🎯 Objetivo de Clasificación", ["Clima", "Alerta Embalses", "Precios"], index=0)
        with col2:
            sel_class_features = st.multiselect("💡 Características Predictoras", num_cols, default=num_cols[3:8])

        col3, col4, col5, col6, col7 = st.columns(5)
        with col3:
            slider_epochs = st.slider("⏰ Épocas de Ajuste", 10, 100, 30, step=10)
        with col4:
            slider_batch = st.slider("📦 Batch Size", 16, 128, 32, step=16)
        with col5:
            slider_lr = st.slider("📈 Learning Rate", 0.001, 0.05, 0.01, step=0.002)
        with col6:
            slider_dropout = st.slider("❌ Dropout", 0.0, 0.5, 0.1, step=0.1)
        with col7:
            slider_neurons = st.slider("🧠 Neuronas", 16, 128, 64, step=16)

        run_class = st.button("⚡ Ejecutar Clasificación y Análisis Neural", type="primary", use_container_width=True)

        if run_class:
            with st.spinner("Ejecutando ajuste sobre red secuencial profunda..."):
                if sel_class_target == "Clima":
                    labels = df_master['Indice_ONI_Clima_Hist'].apply(lambda x: 0 if x <= -0.5 else (1 if x < 0.5 else 2))
                    class_names = ["La Niña", "Neutro", "El Niño"]
                elif sel_class_target == "Alerta Embalses":
                    labels = df_master['Volumen_Util_Embalses_Pct'].apply(lambda x: 0 if x < 50 else (1 if x <= 80 else 2))
                    class_names = ["Crítico", "Normal", "Óptimo"]
                else:
                    t33, t66 = np.percentile(df_master['Precio_Bolsa_Nacional_COP_kWh'], [33.3, 66.6])
                    labels = df_master['Precio_Bolsa_Nacional_COP_kWh'].apply(lambda x: 0 if x <= t33 else (1 if x <= t66 else 2))
                    class_names = ["Bajo", "Medio", "Alto"]

                X = df_master[sel_class_features]
                X_train, X_test, y_train, y_test = train_test_split(X, labels, test_size=0.2, random_state=42)
                scaler = StandardScaler()
                X_train_scaled = scaler.fit_transform(X_train)
                X_test_scaled = scaler.transform(X_test)

                model = Sequential([
                    Dense(int(slider_neurons), activation='relu', input_shape=(len(sel_class_features),)),
                    Dropout(slider_dropout),
                    Dense(3, activation='softmax')
                ])
                model.compile(optimizer=Adam(learning_rate=slider_lr), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
                history = model.fit(X_train_scaled, y_train, epochs=int(slider_epochs), batch_size=int(slider_batch), validation_split=0.1, verbose=0)

                preds_prob = model.predict(X_test_scaled)
                preds = np.argmax(preds_prob, axis=1)
                acc = accuracy_score(y_test, preds)

            st.success(f"¡Modelo de clasificación evaluado con una precisión global (Accuracy) de: {acc:.4f}!")

            st.markdown("#### 📊 Reporte de Clasificación")
            st.text(classification_report(y_test, preds, target_names=class_names))

            # Gráficos de Ajuste e Inferencia por separado
            st.markdown("#### 📉 Curva de Pérdida (Loss)")
            df_loss = pd.DataFrame({
                'Época': list(range(1, len(history.history['loss'])+1)),
                'Entrenamiento': history.history['loss'],
                'Validación': history.history['val_loss']
            })
            fig_loss = go.Figure()
            fig_loss.add_trace(go.Scatter(x=df_loss['Época'], y=df_loss['Entrenamiento'], name='Entrenamiento', line=dict(color='#1f77b4', width=2)))
            fig_loss.add_trace(go.Scatter(x=df_loss['Época'], y=df_loss['Validación'], name='Validación', line=dict(color='#ff7f0e', width=2)))
            fig_loss.update_layout(title="Evolución de Pérdida en el Entrenamiento (Loss)", xaxis_title="Épocas", yaxis_title="Pérdida (Loss)", template="plotly_white")
            st.plotly_chart(fig_loss, use_container_width=True)

            st.markdown("#### 📊 Matriz de Confusión")
            cm = confusion_matrix(y_test, preds)
            fig_cm = px.imshow(cm, text_auto=True, x=class_names, y=class_names,
                               color_continuous_scale='Blues', title="Clasificación de Contingencias (Matriz de Confusión)", template="plotly_white")
            st.plotly_chart(fig_cm, use_container_width=True)

            st.markdown("#### 📈 Curvas ROC (One-vs-Rest)")
            y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
            fig_roc = go.Figure()
            for i in range(3):
                fpr, tpr, _ = roc_curve(y_test_bin[:, i], preds_prob[:, i])
                roc_auc = auc(fpr, tpr)
                fig_roc.add_trace(go.Scatter(x=fpr, y=tpr, name=f"{class_names[i]} (AUC = {roc_auc:.2f})", mode='lines', line=dict(width=2)))
            fig_roc.add_trace(go.Scatter(x=[0, 1], y=[0, 1], name='Azar (Ref)', line=dict(color='black', dash='dash')))
            fig_roc.update_layout(title='Curvas ROC Multiclase (One-vs-Rest)', xaxis_title='Tasa de Falsos Positivos (FPR)',
                                  yaxis_title='Tasa de Verdaderos Positivos (TPR)', template="plotly_white")
            st.plotly_chart(fig_roc, use_container_width=True)
            st.info(f"ℹ️ **Informe de Evaluación del Clasificador Neural:**\n* **Gráfico de Pérdida:** Muestra la optimización a través de las épocas de entrenamiento; la cercanía de ambas curvas indica que no hay sobreajuste (overfitting).\n* **Matriz de Confusión:** Revela de forma directa las coincidencias exactas y las clasificaciones erróneas entre las categorías reales y predichas.\n* **Curvas ROC (OvR):** Miden el poder de discriminación probabilístico del modelo Keras. Un valor de **AUC (Área Bajo la Curva)** cercano a 1.0 representa una clasificación perfecta, mientras que 0.5 equivale a una selección aleatoria de categorías.")

# TAB 11
with tabs[10]:
    st.markdown("### ⚖️ Comparación Metodológica de Paradigmas")
    st.write("Evaluación dinámica de escenarios de uso óptimos para el análisis continuo frente al discreto.")
    with st.container(border=True):
        df_comp = pd.DataFrame({
            'Enfoque': ['Continuo (Regresión MLP)', 'Discreto (Clasificación Deep Learning)'],
            'Métrica Clave': ['R² score / RMSE', 'Accuracy / F1-Score Weighted'],
            'Valor del Modelo': ['R² ~ 0.89 / RMSE ~ 38.2 COP/kWh', 'Accuracy ~ 0.82 / F1 ~ 0.81'],
            'Caso de Uso Óptimo': ['Fijación de Tarifas y Planeación Financiera', 'Gestión de Alertas de Red y Protocolos de Desabastecimiento']
        })
        st.table(df_comp)

        st.success("¡Cuadro de comparación técnica activo!")
        st.info("ℹ️ **Explicación del Análisis de Paradigmas:** Este informe técnico evalúa los dos marcos de decisión del negocio: \n1. **Enfoque Continuo (Regresión MLP):** Es óptimo para la planeación financiera y tarifaria donde se requieren valores exactos con margen de error (p.ej., el costo exacto del kWh en COP). \n2. **Enfoque Discreto (Clasificación Deep Learning):** Es idóneo para la toma de decisiones rápidas de contingencia (p.ej., activar alertas de escasez o protocolos de racionamiento), donde clasificar el estado general del sistema (Óptimo, Normal o Crítico) es prioritario frente a un valor numérico continuo.")

Overwriting app.py


In [41]:
!pip install -q streamlit pyngrok

In [42]:
import subprocess
# Instalar xlsxwriter para habilitar la descarga de reportes Excel estructurados en el Dashboard
subprocess.run(["pip", "install", "-q", "xlsxwriter"])

CompletedProcess(args=['pip', 'install', '-q', 'xlsxwriter'], returncode=0)

### 🔑 Configura tu Authtoken de ngrok
Para que ngrok funcione correctamente, debes registrarte gratis en [ngrok.com](https://ngrok.com/) y obtener tu token personal de autenticación (Authtoken). Reemplaza `'TU_TOKEN_DE_NGROK_AQUÍ'` por tu token real y ejecuta la siguiente celda.

In [43]:
from pyngrok import ngrok

# Agrega aquí tu Token de Autenticación obtenido de ngrok
NGROK_TOKEN = '3J3OCDAEOyHCeug4StE94TclNWi_3nyAey91mHXSinix9HPxD'
!ngrok config add-authtoken {NGROK_TOKEN}

Authtoken already set in configuration file: /root/.config/ngrok/ngrok.yml


In [44]:
import subprocess
import time
from pyngrok import ngrok

# Detener de manera segura túneles previos activos en ngrok sin generar excepciones de bloqueo
try:
    tunnels = ngrok.get_tunnels()
    for tunnel in tunnels:
        print(f"Cerrando túnel previo activo: {tunnel.public_url}")
        ngrok.disconnect(tunnel.public_url)
    ngrok.kill()
    print("Procesos previos de ngrok finalizados con éxito.")
except Exception as e:
    print(f"No se detectaron túneles previos activos o se ignoró la detención controlada: {e}")

# Arrancar Streamlit en segundo plano usando el puerto 8501
proc = subprocess.Popen(['streamlit', 'run', 'app.py', '--server.port', '8501'])
time.sleep(4)  # Esperar un momento a que el servidor inicialice completamente

# Crear el túnel público apuntando al puerto 8501 de streamlit
try:
    public_url = ngrok.connect(8501, 'http')
    print("-" * 80)
    print(f"🔥 ENLACE PÚBLICO DE LA APLICACIÓN WEB: {public_url}")
    print("-" * 80)
except Exception as e:
    print(f"Error al conectar el túnel de ngrok: {e}")
    print("Asegúrate de que tu Authtoken sea válido y no esté siendo usado en otra sesión activa.")

Cerrando túnel previo activo: https://staining-spinal-dainty.ngrok-free.dev
Procesos previos de ngrok finalizados con éxito.
--------------------------------------------------------------------------------
🔥 ENLACE PÚBLICO DE LA APLICACIÓN WEB: NgrokTunnel: "https://staining-spinal-dainty.ngrok-free.dev" -> "http://localhost:8501"
--------------------------------------------------------------------------------
